In [1]:
""" 
Global HRV and PRV metric analysis across all 50 subjects

Script to perform analyses comparing Heart Rate Variability (HRV) and Pulse Rate Variability (PRV) metrics.
This script loads .csv files for all subjects containing metric values for both All Windows and Method 2 
conditions and performs a series of comparative analyses.

Performed analyses:
- Calculation of Pearson and Spearman correlations for each subject
- Violin plots of HRV and PRV metrics across all subjects
- Overall HRV vs PRV metric values across all subjects (mean ± standard deviation)
- Computation of mean, standard deviation (std), coefficient of variation (CV = std/mean), and 
  interquartile range (IQR) of correlations across all subjects
- Bar plots of correlations across all subjects
- Heatmaps of CV and IQR values
- Boxplots of correlation distributions
- Bland-Altman plot 

These analyses were performed considering all 50 subjects and comparing All Windows and Method 2.
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import re
import pandas as pd
import matplotlib.pyplot as plt 


from utils_analysis import (
    compute_correlations_for_metrics, 
    compute_mean_std_metrics,
    prepare_metric_dataframe
)

from utils_analysis_plot import (
    violin_plot,
    plot_correlations_bar, 
    plot_heatmaps_comparison, 
    plot_lollipop_comparison,
    plot_boxplot_correlations,
    plot_bland_altman_all
)

%matplotlib qt  


# =============================================================================
# VARIABLES DEFINITION
# =============================================================================

# In this script, the shorthand notation is used for clarity:
#  - "all" refers to the condition "All Windows"
#  - "m2" refers to the "Method 2" condition, which involves window removal

# Initialize empty DataFrames (for metrics) and list (for correlations) to store aggregated data
global_hrv_all, global_prv_all = pd.DataFrame(), pd.DataFrame()
global_hrv_m2, global_prv_m2 = pd.DataFrame(), pd.DataFrame()
global_pearson_all, global_spearman_all = [], []
global_pearson_m2, global_spearman_m2 = [], []

# Define the base directory to data
base_path = os.path.join("..", "Data")

# Define paths for the two experimental conditions
methods = {
    "All Windows": os.path.join(base_path, "All Windows"),
    "Method 2": os.path.join(base_path, "Window Removal")
}

# List subjects based on HRV folder of 'All Windows'
subjects_dir = os.path.join(methods["All Windows"], "HRV")
subjects = [f for f in os.listdir(subjects_dir) if f.endswith(".csv")]
subjects = sorted(subjects, key=lambda x: int(re.findall(r'\d+', x)[0]))  # sort numerically


# =============================================================================
# LOAD FEATURES AND COMPUTE CORRELATIONS
# =============================================================================

# Note: RMSSD and SD1 are equivalent measures, although on another scale (Ciccone et al., 2017).
#       Therefore, only SD1 was considered in the analysis 

for subj_file in subjects:
    subj_name = subj_file.replace(".csv", "")

    # ------ All Windows ------
    hrv_path = os.path.join(methods["All Windows"], "HRV", f"{subj_name}.csv")
    prv_path = os.path.join(methods["All Windows"], "PRV", f"{subj_name}.csv")

    df_hrv = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
    df_prv = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

    global_hrv_all[f"{subj_name}"] = df_hrv.mean()
    global_prv_all[f"{subj_name}"] = df_prv.mean()

    pearson, spearman = compute_correlations_for_metrics(df_hrv, df_prv)
    global_pearson_all.append(pearson)
    global_spearman_all.append(spearman)


    # ------ Method 2 ------
    hrv_path = os.path.join(methods["Method 2"], "HRV", f"{subj_name}.csv")
    prv_path = os.path.join(methods["Method 2"], "PRV", f"{subj_name}.csv")

    df_hrv = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
    df_prv = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

    global_hrv_m2[f"{subj_name}"] = df_hrv.mean()
    global_prv_m2[f"{subj_name}"] = df_prv.mean()

    pearson, spearman = compute_correlations_for_metrics(df_hrv, df_prv)
    global_pearson_m2.append(pearson)
    global_spearman_m2.append(spearman)


In [2]:
# =============================================================================
# VIOLIN PLOT ON METRICS
# =============================================================================

violin_plot(global_hrv_all, global_prv_all,
            global_hrv_m2, global_prv_m2)
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/violin_plot_v3.png", dpi=600, bbox_inches='tight') 


In [3]:
# =============================================================================
# COMPUTE MEAN ± STD FOR EACH METRIC ACROSS ALL SUBJECTS
# =============================================================================

# ------ All Windows ------
global_metrics = pd.concat([
    compute_mean_std_metrics(global_hrv_all, "HRV"),
    compute_mean_std_metrics(global_prv_all, "PRV")
], axis=1)


# ------ Method 2 ------
global_metrics_m2 = pd.concat([
    compute_mean_std_metrics(global_hrv_m2, "HRV"),
    compute_mean_std_metrics(global_prv_m2, "PRV")
], axis=1)


print("\n HRV/PRV METRICS: ALL WINDOWS")
print(global_metrics)

print("\n HRV/PRV METRICS: METHOD 2")
print(global_metrics_m2)



 HRV/PRV METRICS: ALL WINDOWS
                               HRV                 PRV
HRV_MeanNN      1033.296 ± 159.566  1035.280 ± 159.055
HRV_SDNN           69.101 ± 33.232     75.979 ± 31.642
HRV_LFn              0.354 ± 0.105       0.352 ± 0.103
HRV_HFn              0.334 ± 0.143       0.367 ± 0.130
HRV_LFHF             2.320 ± 1.960       1.849 ± 1.399
HRV_VLF              0.010 ± 0.002       0.010 ± 0.002
HRV_LF               0.017 ± 0.006       0.020 ± 0.007
HRV_HF               0.021 ± 0.015       0.025 ± 0.015
HRV_ApEn             0.960 ± 0.102       0.982 ± 0.096
HRV_SampEn           1.316 ± 0.277       1.356 ± 0.289
HRV_DFA_alpha1       0.977 ± 0.254       0.889 ± 0.221
HRV_DFA_alpha2       0.915 ± 0.173       0.855 ± 0.155
HRV_SD1            48.696 ± 33.709     58.238 ± 31.762
HRV_SD2            82.097 ± 35.940     88.024 ± 34.599
HRV_SD1SD2           0.578 ± 0.225       0.660 ± 0.205

 HRV/PRV METRICS: METHOD 2
                               HRV                 PRV
HRV_Me

In [4]:
# =============================================================================
# COMPUTE MEAN, STD, COEFFICIENT OF VARIATION (CV) 
# AND RANGE INTERQUARTILE (IQR) OF CORRELATIONS
# =============================================================================

# ------ All Windows Correlations ------
df_pearson_global = pd.DataFrame(global_pearson_all)
df_spearman_global = pd.DataFrame(global_spearman_all)

global_pearson_all_mean = df_pearson_global.mean().to_dict()
global_pearson_all_std = df_pearson_global.std().to_dict()
global_spearman_all_mean = df_spearman_global.mean().to_dict()
global_spearman_all_std = df_spearman_global.std().to_dict()

CV_pearson_all = {k: global_pearson_all_std[k] / global_pearson_all_mean[k] for k in global_pearson_all_mean}
CV_spearman_all = {k: global_spearman_all_std[k] / global_spearman_all_mean[k] for k in global_spearman_all_mean}

iqr_pearson_all = (df_pearson_global.quantile(0.75) - df_pearson_global.quantile(0.25)).to_dict()
iqr_spearman_all = (df_spearman_global.quantile(0.75) - df_spearman_global.quantile(0.25)).to_dict()


# ------ Method 2 Correlations ------
df_pearson_global_m2 = pd.DataFrame(global_pearson_m2)
df_spearman_global_m2 = pd.DataFrame(global_spearman_m2)

global_pearson_m2_mean = df_pearson_global_m2.mean().to_dict()
global_pearson_m2_std = df_pearson_global_m2.std().to_dict()
global_spearman_m2_mean = df_spearman_global_m2.mean().to_dict()
global_spearman_m2_std = df_spearman_global_m2.std().to_dict()

CV_pearson_m2 = {k: global_pearson_m2_std[k] / global_pearson_m2_mean[k] for k in global_pearson_m2_mean}
CV_spearman_m2 = {k: global_spearman_m2_std[k] / global_spearman_m2_mean[k] for k in global_spearman_m2_mean}

iqr_pearson_m2  = (df_pearson_global_m2.quantile(0.75) - df_pearson_global_m2.quantile(0.25)).to_dict()
iqr_spearman_m2  = (df_spearman_global_m2.quantile(0.75) - df_spearman_global_m2.quantile(0.25)).to_dict()


In [5]:
# =============================================================================
# BARPLOT OF GROUPED PEARSON AND SPEARMAN CORRELATIONS (WITH STD)
# =============================================================================

plot_correlations_bar(
    global_pearson_all_mean, global_spearman_all_mean,
    global_pearson_m2_mean, global_spearman_m2_mean,
    global_pearson_all_std, global_spearman_all_std,
    global_pearson_m2_std, global_spearman_m2_std,
    std_plot=True
)
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/barplot_correlations_v3.png", dpi=600, bbox_inches='tight') 


In [6]:
# =============================================================================
# CV AND IQR HEATMAPS AND LOLLIPOP COMPARISON
# =============================================================================

# Prepare DataFrames
df_cv_pearson = prepare_metric_dataframe(CV_pearson_all, CV_pearson_m2)
df_cv_spearman = prepare_metric_dataframe(CV_spearman_all, CV_spearman_m2)

df_iqr_pearson = prepare_metric_dataframe(iqr_pearson_all, iqr_pearson_m2)
df_iqr_spearman = prepare_metric_dataframe(iqr_spearman_all, iqr_spearman_m2)


### LOLLIPOP
plot_lollipop_comparison(df_cv_pearson, df_cv_spearman, "CV")
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/CV_lollipop_v3.png", dpi=600, bbox_inches='tight') 

plot_lollipop_comparison(df_iqr_pearson, df_iqr_spearman, "IQR")
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/IQR_lollipop_v3.png", dpi=600, bbox_inches='tight') 


### HEATMAPS
plot_heatmaps_comparison(df_cv_pearson, df_cv_spearman,
                         title_prefix="CV", cmap="coolwarm")

plot_heatmaps_comparison(df_iqr_pearson, df_iqr_spearman,
                         title_prefix="IQR", cmap="YlGnBu")

c:\Users\ilari\Documents\GitHub\HRVvsPRV\03. Analysis\utils_analysis_plot.py:301: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])
c:\Users\ilari\Documents\GitHub\HRVvsPRV\03. Analysis\utils_analysis_plot.py:301: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 1])


In [7]:
# =============================================================================
# BOXPLOT OF CORRELATIONS 
# =============================================================================

plot_boxplot_correlations(df_pearson_global, df_pearson_global_m2,
                          df_spearman_global, df_spearman_global_m2)
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/boxplot_v3.png", dpi=600, bbox_inches='tight') 


In [ ]:
# =============================================================================
# BLAND-ALTMAN PLOT 
# =============================================================================

plot_bland_altman_all(global_hrv_all.T, global_prv_all.T, global_hrv_m2.T, global_prv_m2.T)
#plt.savefig("C:/Users/ilari/Documents/GitHub/HRVvsPRV/Img and Results/bland_altman_allWindows_method2_v3.png", dpi=600, bbox_inches='tight') 
